In [1]:
import os

# Set working directory to project root always`n
# Works regardless of where the notebook is saved`n
notebook_dir = os.path.dirname(os.path.abspath('__file__'))
project_root = os.path.abspath(os.path.join(notebook_dir, '..'))
os.chdir(project_root)
print("Working directory set to:", os.getcwd())

Working directory set to: c:\Users\DELL\OneDrive\Documents\SRM\Churn_Analysis


In [2]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder

df = pd.read_csv(
    'data/telecom/WA_Fn-UseC_-Telco-Customer-Churn.csv'
)

# Fix dtype bug
df['TotalCharges'] = pd.to_numeric(
    df['TotalCharges'], errors='coerce'
)

# Drop unique identifier â€” no predictive value
df.drop(columns=['customerID'], inplace=True)

# Fill 11 TotalCharges nulls with 0 (new customers)
df['TotalCharges'].fillna(0, inplace=True)

# Label encode target
df['Churn'] = df['Churn'].map({'Yes':1, 'No':0})

# Binary Yes/No columns
binary_cols = [
    'Partner','Dependents','PhoneService',
    'PaperlessBilling','MultipleLines',
    'OnlineSecurity','OnlineBackup',
    'DeviceProtection','TechSupport',
    'StreamingTV','StreamingMovies'
]
for col in binary_cols:
    df[col] = df[col].map({
        'Yes':1, 'No':0,
        'No phone service':0,
        'No internet service':0
    })

df['gender'] = df['gender'].map({'Male':1,'Female':0})

# One-hot encode multi-class columns
df = pd.get_dummies(
    df,
    columns=['Contract','InternetService','PaymentMethod'],
    drop_first=True
)

print("Shape after encoding:", df.shape)
print("Nulls remaining     :", df.isnull().sum().sum())
print("Dtypes left as object:")
print(df.select_dtypes('object').columns.tolist())

Shape after encoding: (7043, 24)
Nulls remaining     : 11
Dtypes left as object:
[]


C:\Users\DELL\AppData\Local\Temp\ipykernel_17080\3081333048.py:17: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df['TotalCharges'].fillna(0, inplace=True)


In [3]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# Separate features and target
X = df.drop(columns=['Churn'])
y = df['Churn']

# Scale numerical columns only
scaler = StandardScaler()
scale_cols = ['tenure','MonthlyCharges','TotalCharges']
X[scale_cols] = scaler.fit_transform(X[scale_cols])

# Train-test split â€” stratify keeps churn ratio equal
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("\nChurn rate â€” train:", round(y_train.mean(),3))
print("Churn rate â€” test :", round(y_test.mean(),3))

X_train: (5634, 23)
X_test : (1409, 23)

Churn rate â€” train: 0.265
Churn rate â€” test : 0.265


In [4]:
import joblib
import os

os.makedirs('outputs', exist_ok=True)
joblib.dump(scaler, 'outputs/scaler_telecom.pkl')
print("Scaler saved")

Scaler saved
